## Calibration of the IAM 

### Simulation fn. 

The following attributes are required to have in  a dataframe in order to use/calibrate the IAM. 

#### 1. Core Ego-Vehicle Columns

| Column Name            | Description                               |
|------------------------|-------------------------------------------|
| `Speed [km/h]`         | Ego longitudinal speed (km/h)             |
| `Speed [km/h]_lat`     | Ego lateral speed                          |
| `Tan. Acc. [ms-2]`     | Ego longitudinal acceleration             |
| `x_rot`                | Ego x-position (rotated/global)           |
| `y_rot`                | Ego y-position (rotated/global)           |
| `width`                | Ego vehicle width                         |
| `lanes`                | Ego lane index                            |
| `x_leader`             | X-position of the immediate leader        |

---

#### 2. Road / Lane Geometry Columns

| Column Name     | Description                              |
|------------------|------------------------------------------|
| `width_left`     | Distance to left road boundary           |
| `width_right`    | Distance to right road boundary          |

---

#### 3. Neighbor Vehicle Columns (Vehicles 1–6)

#### Followers (1, 2, 3)

| Column Name              | Description                        |
|---------------------------|------------------------------------|
| `Speed [km/h]_1`         | Follower 1 speed                   |
| `Tan. Acc. [ms-2]_1`     | Follower 1 longitudinal accel.     |
| `width_1`                | Follower 1 width                   |
| `Speed [km/h]_2`         | Follower 2 speed                   |
| `Tan. Acc. [ms-2]_2`     | Follower 2 longitudinal accel.     |
| `width_2`                | Follower 2 width                   |
| `Speed [km/h]_3`         | Follower 3 speed                   |
| `Tan. Acc. [ms-2]_3`     | Follower 3 longitudinal accel.     |
| `width_3`                | Follower 3 width                   |

#### Leaders (4, 5, 6)

| Column Name              | Description                        |
|---------------------------|------------------------------------|
| `Speed [km/h]_4`         | Leader 4 speed                     |
| `Tan. Acc. [ms-2]_4`     | Leader 4 longitudinal accel.       |
| `width_4`                | Leader 4 width                     |
| `Speed [km/h]_5`         | Leader 5 speed                     |
| `Tan. Acc. [ms-2]_5`     | Leader 5 longitudinal accel.       |
| `width_5`                | Leader 5 width                     |
| `Speed [km/h]_6`         | Leader 6 speed                     |
| `Tan. Acc. [ms-2]_6`     | Leader 6 longitudinal accel.       |
| `width_6`                | Leader 6 width                     |


#### Simulation Logic 

- Initialize speeds, positions, accelerations, deviations, and lane-change counters.
- For each timestep:
  - Update longitudinal and lateral speeds using previous accelerations.
  - Update positions using trapezoidal integration (mean speed).
  - Prevent backward motion by enforcing x[i] ≥ x[i−1].
  - Enforce minimum gap to leader; correct longitudinal position if too close.
  - If no real lane changes exist:
    - Prevent unwanted lane changes by resetting if lateral deviation > 2 m.
  - If one real lane change exists:
    - Check if simulated lane change begins within tolerance.
    - If not, count as a missing lane change.
  - If two real lane changes exist:
    - Check both lane-change time windows for missing lateral movement.
  - Compute deviation from ground-truth (Euclidean, longitudinal, lateral).
  - Recalculate relative distances to neighbors.
  - Compute longitudinal acceleration components:
    - Free-road IDM acceleration.
    - Leader pull (min of interactions with vehicles 4–6).
    - Follower push (anisotropic interactions with vehicles 1–3).
    - Road-boundary longitudinal term.
  - Compute lateral acceleration components:
    - Free lateral relaxation.
    - Leader lateral interactions.
    - Follower lateral interactions (anisotropic).
    - Boundary repulsion force.
    - Flow-field correction toward lane center.
  - Store updated accelerations for next step.
- After simulation completes:
  - Compute RMSE errors for total, longitudinal, and lateral deviation.
  - Compute weighted real error (penalizes lateral deviation heavily).
  - Add penalties for unwanted or missed lane changes.
  - Return simulation metrics and updated DataFrame.
 
####  Summary

| Category                     | Purpose                                      |
|-----------------------------|----------------------------------------------|
| Position integration        | Compute new vehicle position/speed           |
| Negative motion prevention  | Avoid backward motion                        |
| Collision avoidance         | Enforce minimum gap to leader                |
| Lane-change correctness     | Penalize unwanted or missed lane changes     |
| Boundary enforcement        | Repel vehicle from road edges                |
| Interaction forces          | Leader/follower forces (longitudinal & lateral) |
| Flow field                  | Encourage lane-center following              |
| Deviation metrics           | Compare simulation to real trajectory        |
| GOF score                   | Combine geometric + behavioral accuracy      |


In [ ]:
def sim(s0y, s0b, lat_sens, relax_time, fb, gb, lc_param, aniso,  data):
    
    GAP_MIN = 0.4
    data = data.reset_index()
    dt = 0.2

    data['lane_change'] = data['lanes'].ne(data['lanes'].shift())
    change_indexes = data.index[data['lane_change']].tolist()
    number_of_lane_changings = len(change_indexes)
    
    # IAM instance declaration. 
    IAM_model = IAM(IDM, s0y, s0b, lat_sens, relax_time, fb, gb, lc_param)

    # create the array to save the new information 

    v1=np.empty(len(data), dtype=float)
    v2=np.empty(len(data), dtype=float)
    v1[0] = data.loc[0, 'Speed [km/h]'] * 0.2777778 # first value of speed is taken 
    v2[0] = data.loc[0, 'Speed [km/h]_lat'] 

    x1=np.empty(len(data), dtype=float)
    y2=np.empty(len(data), dtype=float)
    x1[0] = data.loc[0, 'x_rot']  # fill the first position from data. 
    y2[0] = data.loc[0, 'y_rot'] 

    x1_true = data['x_rot'].values # for the existing x and y values to calculate the devaition 
    y2_true = data['y_rot'].values 
    deviation = np.empty(len(data), dtype=float)
    deviation_lat =  np.empty(len(data), dtype=float)
    deviation_long =  np.empty(len(data), dtype=float)
    leader_distance = np.empty(len(data), dtype=float)
    deviation[0] = 0 # first point devivation is 0. 
    deviation_lat[0] = 0 
    deviation_long[0] = 0 
  
    acc1=np.empty(len(data), dtype=float) # long. acc. 
    acc2=np.empty(len(data), dtype=float) # lat. acc. 

    #### 
    unwanted_lane_change_count = 0 
    lane_changing_count = 0 
    missing_lane_changes_count = 0 

    # initialize the input data for the first acceleration prediction. 
    ####################################################################
    width_left = data.loc[0, 'width_left']
    width_right = data.loc[0, 'width_right'] 
    vxo = 31.008277 # the desired speed from the IDM. 
    Wveh =  data.loc[0, 'width'] 
    vy = data.loc[0, 'Speed [km/h]_lat'] # this is ms-2 hence no need to turn 
    vx =  data.loc[0, 'Speed [km/h]']* 0.2777778
    ax =  data.loc[0, 'Tan. Acc. [ms-2]']

    # variable calculation 
    df_row = data.loc[0,]
    dx_1, dx_2, dx_3, dx_4, dx_5, dx_6, dy_1, dy_2, dy_3, dy_4, dy_5, dy_6 = variable_calc(x1[0], y2[0],  df_row)
    
    vxl_4, axl_4, Wavg_4 = df_row['Speed [km/h]_4']* 0.2777778, df_row['Tan. Acc. [ms-2]_4'], 0.5*(Wveh + df_row['width_4'])
    vxl_5, axl_5, Wavg_5 = df_row['Speed [km/h]_5']* 0.2777778, df_row['Tan. Acc. [ms-2]_5'], 0.5*(Wveh + df_row['width_5'])
    vxl_6, axl_6, Wavg_6 = df_row['Speed [km/h]_6']* 0.2777778, df_row['Tan. Acc. [ms-2]_6'], 0.5*(Wveh + df_row['width_6'])

    vxl_1,  Wavg_1 = df_row['Speed [km/h]_1']* 0.2777778, 0.5*(Wveh + df_row['width_1'])
    vxl_2,  Wavg_2 = df_row['Speed [km/h]_2']* 0.2777778, 0.5*(Wveh + df_row['width_2'])
    vxl_3,  Wavg_3 = df_row['Speed [km/h]_3']* 0.2777778, 0.5*(Wveh + df_row['width_3'])

    # long acc calculation (1st step) 
    ####################################################################################################################################
    long_free = IDM.calc_acc_free(vx)

    first_leader_pull = IAM_model.calc_acc_long_int(dx_4, dy_4, vx, vxl_4, axl_4, Wavg_4)
    second_leader_pull = IAM_model.calc_acc_long_int(dx_5, dy_5, vx, vxl_5, axl_5, Wavg_5)
    third_leader_pull = IAM_model.calc_acc_long_int(dx_6, dy_6, vx, vxl_6, axl_6, Wavg_6)
    pull_long = min(first_leader_pull, second_leader_pull, third_leader_pull)    

    # note that the variables are changed according to the change of leader and follower since this is for the followers. 
    first_follower_push = aniso*IAM_model.calc_acc_long_int(dx_1, dy_1, vxl_1, vx, ax, Wavg_1)
    second_follower_push = aniso*IAM_model.calc_acc_long_int(dx_2, dy_2, vxl_2, vx, ax, Wavg_2)
    third_follower_push = aniso*IAM_model.calc_acc_long_int(dx_3, dy_3, vxl_3, vx, ax, Wavg_3)
    push_long = max(first_follower_push, second_follower_push, third_follower_push)

    long_boundary, lat_boundary = IAM_model.calc_acc_b(width_left , width_right, vx, vxo, Wveh)
    
    long_acc =  np.nansum([long_free, pull_long, push_long, long_boundary])
    ########################################################################################################################################

    # lat. acc. calculation (1st step)
    #######################################################################################################
    lat_free  = IAM_model.calc_acc_lat_free(vy) 

    first_leader_lat_int = IAM_model.calc_acc_lat_int(dx_4, dy_4, vx, vxl_4, vy, axl_4 , Wavg_4)
    second_leader_lat_int = IAM_model.calc_acc_lat_int(dx_5, dy_5, vx, vxl_5, vy, axl_5 , Wavg_5)
    third_leader_lat_int = IAM_model.calc_acc_lat_int(dx_6, dy_6, vx, vxl_6, vy, axl_6 , Wavg_6)
    leader_lat_int_list = [first_leader_lat_int, second_leader_lat_int, third_leader_lat_int]
    lat_int_leaders = np.nansum(leader_lat_int_list)

    first_follower_lat_int = -aniso*(IAM_model.calc_acc_lat_int(dx_1, dy_1, vxl_1, vx,  vy, ax , Wavg_1)) 
    second_followr_lat_int = -aniso*(IAM_model.calc_acc_lat_int(dx_2, dy_2, vxl_2, vx,  vy, ax , Wavg_2))
    third_follower_lat_int = -aniso*(IAM_model.calc_acc_lat_int(dx_3, dy_3, vxl_3, vx,  vy, ax , Wavg_3))
    follower_lat_int_list = [first_follower_lat_int , second_followr_lat_int, third_follower_lat_int]
    lat_int_followers =  np.nansum(follower_lat_int_list)

    flow_feild = IAM_model.calc_ff(ff_width_calc(y2[0]))

    lat_acc = np.nansum([lat_free, lat_int_followers, lat_int_leaders, lat_boundary, flow_feild])
    ##########################################################################################################

    # filling the information for the first step 
    acc1[0] = long_acc
    acc2[0] = lat_acc

    # simulation
    for i in range(1,len(data)):

        v1[i] = v1[i-1] + acc1[i-1]*dt 
        v2[i] = v2[i-1] + acc2[i-1]*dt  

        # calculation of the new posititions. 
        x1[i] =  0.5*(v1[i]+v1[i-1])*dt + x1[i-1] # take the mean speed to calcualte the distance and then add the previous position to take the new position. 
        y2[i] =  0.5*(v2[i]+v2[i-1])*dt + y2[i-1]

        if x1[i] < x1[i-1]: ## solve if any negative movememnt occurs. 
            
            x1[i] = x1[i-1] + 0.4 
        
        #######################################################################################################################################
        ### check for long. colapse and avoid it.

        leader_gap = data.loc[i, 'x_leader'] -  x1[i]

        if leader_gap < GAP_MIN: 

            #x1[i] = data.loc[i, 'x_rot']

            x1[i] = data.loc[i, 'x_leader'] - 0.4

        
        ## ################################################################################################################################################
        # lane changing consideration 
        # wrongful changings for vehicles without lane changing. 
        if number_of_lane_changings == 1:  # no lane changes. 
            
            lat_gap = abs(y2[i] - y2[0])
            
            if lat_gap > 2: 
    
                x1[i] = data.loc[i, 'x_rot']
                y2[i] = data.loc[i, 'y_rot']
    
                unwanted_lane_change_count += 1 

        #  missing the lane changes 
        ## for only one lane changing 
        if number_of_lane_changings == 2:   

            if change_indexes[0] + 20 > len(data): 
                gap = len(data) - change_indexes[0]
            else: 
                gap = 20 

            if i - gap == change_indexes[0]:       ## 3 second tolerance period

                lat_gap = abs(y2[i] - y2[0])

                if abs(lat_gap) < 2: 
    
                    x1[i] = data.loc[i, 'x_rot']
                    y2[i] = data.loc[i, 'y_rot']  
    
                    missing_lane_changes_count  += 1 

                else: 
                    
                    lane_changing_count +=1
                    
        ## for two lane changing 
        if number_of_lane_changings == 3:  

            if change_indexes[1] + 20 > len(data): 
                gap_2 = len(data) - change_indexes[1]
            else: 
                gap_2 = 20            

            if i - 20 == change_indexes[0]:       ## 3 second tolerance period
    
                lat_gap = y2[i] - y2[0]
    
                if abs(lat_gap) < 2: 
    
                    x1[i] = data.loc[i, 'x_rot']
                    y2[i] = data.loc[i, 'y_rot']  
    
                    missing_lane_changes_count  += 1 
    
                else: 
                    
                    lane_changing_count +=1

            
            if i - gap_2 == change_indexes[1]:       ## 4 second tolerance period

                lat_gap = y2[i] - y2[change_indexes[0] + 20]
        
                if abs(lat_gap) < 2: 
        
                    x1[i] = data.loc[i, 'x_rot']
                    y2[i] = data.loc[i, 'y_rot']  
        
                    missing_lane_changes_count  += 1 
        
                else: 
                    
                    lane_changing_count +=1

                
        #################################################################################################################################################

        # calcualte the gap and add it 
        deviation[i] = euc_gap_calc(x1[i], y2[i], x1_true[i], y2_true[i])
        deviation_lat[i] =  y2[i] - y2_true[i]
        deviation_long[i] = x1[i] - x1_true[i]
  
       
        # check for the colitions and make any changes if necessary. 
        # for now we ignore the colitions. 

        df_row = data.loc[i,]
        # calculate the new relative variables.# we assume the neighbors does not change.  
        dx_1, dx_2, dx_3, dx_4, dx_5, dx_6, dy_1, dy_2, dy_3, dy_4, dy_5, dy_6 = variable_calc(x1[i], y2[i],  df_row)
        
        vxl_4, axl_4, Wavg_4 = df_row['Speed [km/h]_4']* 0.2777778, df_row['Tan. Acc. [ms-2]_4'], 0.5*(Wveh + df_row['width_4'])
        vxl_5, axl_5, Wavg_5 = df_row['Speed [km/h]_5']* 0.2777778, df_row['Tan. Acc. [ms-2]_5'], 0.5*(Wveh + df_row['width_5'])
        vxl_6, axl_6, Wavg_6 = df_row['Speed [km/h]_6']* 0.2777778, df_row['Tan. Acc. [ms-2]_6'], 0.5*(Wveh + df_row['width_6'])

        vxl_1,  Wavg_1 = df_row['Speed [km/h]_1']* 0.2777778, 0.5*(Wveh + df_row['width_1'])
        vxl_2,  Wavg_2 = df_row['Speed [km/h]_2']* 0.2777778, 0.5*(Wveh + df_row['width_2'])
        vxl_3,  Wavg_3 = df_row['Speed [km/h]_3']* 0.2777778, 0.5*(Wveh + df_row['width_3'])
        ax             = df_row['Tan. Acc. [ms-2]']

        # calculate the new gap to the road boundary. 
        width_left = y2[i] - 86
        width_right = 101.5 - y2[i]
        

        # long. acc. calculation 
        ####################################################################################################################################
        long_free = IDM.calc_acc_free(v1[i])
    
        first_leader_pull = IAM_model.calc_acc_long_int(dx_4, dy_4, v1[i], vxl_4, axl_4, Wavg_4)
        second_leader_pull = IAM_model.calc_acc_long_int(dx_5, dy_5, v1[i], vxl_5, axl_5, Wavg_5)
        third_leader_pull = IAM_model.calc_acc_long_int(dx_6, dy_6, v1[i], vxl_6, axl_6, Wavg_6)
        pull_long = min(first_leader_pull, second_leader_pull, third_leader_pull)    
    
        # note that the variables are changed according to the change of leader and follower since this is for the followers. 
        first_follower_push = aniso*IAM_model.calc_acc_long_int(dx_1, dy_1, vxl_1, v1[i], ax, Wavg_1)
        second_follower_push = aniso*IAM_model.calc_acc_long_int(dx_2, dy_2, vxl_2, v1[i], ax, Wavg_2)
        third_follower_push = aniso*IAM_model.calc_acc_long_int(dx_3, dy_3, vxl_3, v1[i], ax, Wavg_3)
        push_long = max(first_follower_push, second_follower_push, third_follower_push)
    
        long_boundary, lat_boundary = IAM_model.calc_acc_b(width_left, width_right, v1[i], vxo, Wveh)
        
        long_acc = np.nansum([long_free, pull_long, push_long, long_boundary])
        ########################################################################################################################################

        # lat. acc. calculation 
        #######################################################################################################
        lat_free  = IAM_model.calc_acc_lat_free(v2[i]) 
    
        first_leader_lat_int = IAM_model.calc_acc_lat_int(dx_4, dy_4, v1[i], vxl_4, v2[i], axl_4 , Wavg_4)
        second_leader_lat_int = IAM_model.calc_acc_lat_int(dx_5, dy_5, v1[i], vxl_5, v2[i], axl_5 , Wavg_5)
        third_leader_lat_int = IAM_model.calc_acc_lat_int(dx_6, dy_6, v1[i], vxl_6, v2[i], axl_6 , Wavg_6)
        leader_lat_int_list = [first_leader_lat_int, second_leader_lat_int, third_leader_lat_int]
        lat_int_leaders = np.nansum(leader_lat_int_list)
    
    
        first_follower_lat_int = -aniso*(IAM_model.calc_acc_lat_int(dx_1, dy_1, vxl_1, v1[i],  v2[i], ax , Wavg_1)) 
        second_followr_lat_int = -aniso*(IAM_model.calc_acc_lat_int(dx_2, dy_2, vxl_2, v1[i],  v2[i], ax , Wavg_2))
        third_follower_lat_int = -aniso*(IAM_model.calc_acc_lat_int(dx_3, dy_3, vxl_3, v1[i],  v2[i], ax , Wavg_3))
        follower_lat_int_list = [first_follower_lat_int , second_followr_lat_int, third_follower_lat_int]
        lat_int_followers =  np.nansum(follower_lat_int_list)

        flow_feild = IAM_model.calc_ff(ff_width_calc(y2[i]))
    
        lat_acc = np.nansum([lat_free, lat_int_followers, lat_int_leaders, lat_boundary, flow_feild])
        ##########################################################################################################

        acc1[i] = long_acc
        acc2[i] = lat_acc

    # fill the new data to the df. 
    data['v1']=v1.tolist()
    data['v2']=v2.tolist()
    data['x1']=x1.tolist()
    data['y2']=y2.tolist()
    data['deviation']=deviation.tolist()
    data['deviation_lat'] = deviation_lat.tolist()
    data['deviation_long'] = deviation_long.tolist()
    data['acc1']=acc1.tolist()
    data['acc2']=acc2.tolist()
    
    # sum the the squared gap. 
    sse = np.sum(deviation**2)
    avg_error = np.sqrt(sse/len(data))

    sse_long = np.sum(deviation_long**2)
    avg_error_long = np.sqrt(sse_long/len(data))

    sse_lat = np.sum(deviation_lat**2)
    avg_error_lat = np.sqrt(sse_lat/len(data))

    real_sse = sse_long + 100*sse_lat
    real_avg_error = np.sqrt(real_sse)

    real_gof = real_avg_error + 0.5*(unwanted_lane_change_count + missing_lane_changes_count)

    # create the return values. 
    return avg_error, real_avg_error,  avg_error, avg_error_long,  avg_error_lat, missing_lane_changes_count, lane_changing_count, unwanted_lane_change_count , data


### Sim. Related Functions 

#### Simulation-Related Helper Functions — Description

Below are the supporting functions used around the main `sim()` function.  
These functions handle data extraction, parameter normalization, calibration, optimization, and result formatting.

- **data_sim()** → Selects and cleans follower-specific data.  
- **normparams() / denormparams()** → Handle scaling of parameters for optimization.  
- **calib()** → Objective function for Nelder–Mead (returns simulation error).  
- **optim()** → Performs calibration for one follower.  
- **optim_results()** → Aggregates calibrated parameters and simulation scores.


In [1]:
## simulation related functions 
#####################################################################################################################################################
def data_sim(data,follower):
    
    d1=data[follower]
    d1=d1.dropna(subset=['vehicle_id'])

    return d1
    

def normparams(x0):
    
    x1=[0,0,0,0,0,0,0,0]
    
    x1[0] = (x0[0]-0.05)/4
    x1[1] = (x0[1]-0.05)/4
    x1[2] = (x0[2]-0.05)/1
    x1[3] = (x0[3]-0.05)/3
    x1[4] = (x0[4]-0.05)/3
    x1[5] = (x0[5]-0.05)/9
    x1[6] = (x0[6]-0.05)/1
    x1[7] = (x0[7]-0.05)/1
    
    return x1

def denormparams(x1):
    
    x0=[0,0,0,0,0,0,0,0]
    
    x0[0]=np.minimum(np.maximum((x1[0]*4+0.05),0.05),4)  
    x0[1]=np.minimum(np.maximum((x1[1]*10+0.05),0.05),4) 
    x0[2]=np.minimum(np.maximum((x1[2]*5+0.05),0.05),1) 
    x0[3]=np.minimum(np.maximum((x1[3]*5+0.05),0.05),3) 
    x0[4]=np.minimum(np.maximum((x1[4]*5+0.05),0.05),3) 
    x0[5]=np.minimum(np.maximum((x1[5]*9+0.05),0.05),9) 
    x0[6]=np.minimum(np.maximum((x1[6]*2+0.05),0.05),1) 
    x0[7]=np.minimum(np.maximum((x1[7]*1+0.05),0.05),1) 
    
    return x0


def calib(x,data,follower):
    
    x0 = denormparams(x)
    
    s0y = np.minimum(np.maximum(x0[0],0.05),4)
    s0b = np.minimum(np.maximum(x0[1],0.05),4)
    lat_sens = np.minimum(np.maximum(x0[2],0.05),1)
    relax_time = np.minimum(np.maximum(x0[3],0.05),3)
    fb = np.minimum(np.maximum(x0[4],0.05),3)
    gb = np.minimum(np.maximum(x0[5],0.05),9)
    lc_param = np.minimum(np.maximum(x0[6],0.05),1)   
    aniso = np.minimum(np.maximum(x0[7],0.05),1) 

    return sim(s0y, s0b, lat_sens, relax_time, fb, gb, lc_param, aniso, data_sim(data,follower))[0]



def optim(x,data_ideal,follower):

    #optim function to be used in parallerisation using Nelder-Mead method
    res = minimize(calib, x, (data_ideal,follower), method='Nelder-Mead')
    
    return res.x


def optim_results(data_ideal,result,followers):
    
    d={'Follower':[],'s0y':[],'s0b':[],'lat_sens':[],'relax_time':[],'fb':[], 'gb':[],'lc_param':[], 'aniso':[],'SSE':[],'GOF':[]}

    
    for _ in range(len(result)):

        results_denorm = denormparams((result[_])) 
        
        d['Follower'].append(followers[_])
        
        d['s0y'].append(results_denorm[0])
        d['s0b'].append(results_denorm[1])
        d['lat_sens'].append(results_denorm[2])
        d['relax_time'].append(results_denorm[3])
        d['fb'].append(results_denorm[4])
        d['gb'].append(results_denorm[5])
        d['lc_param'].append(results_denorm[6])
        d['aniso'].append(results_denorm[7])
        
        d['SSE'].append(sim(*(results_denorm),data_sim(data_ideal,followers[_]))[0])
        d['GOF'].append(sim(*(results_denorm),data_sim(data_ideal,followers[_]))[1])
    
    return d

### Calibration 

In [ ]:
import time 
from scipy.optimize import minimize, rosen
from joblib import Parallel, delayed
import multiprocessing

data_list = dfs[:50]

x = normparams((0.3, 0.2, 1, 1, 0.2, 5, 1, 0.2)) # Initiaial Parameters have been given

num_cores=4
st=time.time()

print(f'Optimisation started for {len(data_list)} vehicels using {num_cores} cores ')

result = Parallel(n_jobs=num_cores)(delayed(optim)(x=x, data_ideal=data_list ,follower=f) for f in range(len(data_list )))

results = optim_results(data_list ,result,range(len(data_list )))